# Multi-Agent College Mental Health Analysis - Quick Start

This notebook demonstrates the basic usage of the multi-agent system for analyzing college mental health data.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

from src.utils.logger import setup_logger
from src.utils.config_loader import load_config
from src.knowledge_graph.graph_builder import KnowledgeGraphBuilder
from src.knowledge_graph.graph_query import GraphQuery
from src.agents.orchestrator import OrchestratorAgent
from src.agents.spatial_agent import SpatialAgent
from src.agents.behavioral_agent import BehavioralAgent
from src.agents.mental_health_agent import MentalHealthAgent
from src.agents.base_agent import AgentMessage

# Setup logging
setup_logger(level="INFO")

print("✓ Imports successful!")

## 2. Load Configuration

In [ ]:
config = load_config('../config/config.yaml')
print(f"LLM Provider: {config.llm.provider}")
print(f"Primary Model: {config.llm.models['primary']}")
print(f"Knowledge Graph Backend: {config.knowledge_graph.backend}")

## 3. Initialize Knowledge Graph (if available)

In [ ]:
from pathlib import Path

graph_builder = KnowledgeGraphBuilder(backend="networkx")

graph_path = Path("../data/knowledge_graph.graphml")
if graph_path.exists():
    graph_builder.load_graph(graph_path)
    print("✓ Knowledge graph loaded!")
    
    stats = graph_builder.get_statistics()
    print(f"\nGraph Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value}")
else:
    print("⚠ Knowledge graph not found. Run main.py --mode build-graph first.")

graph_query = GraphQuery(graph_builder.graph)

## 4. Initialize Agents

In [ ]:
# Create specialized agents
spatial_agent = SpatialAgent(
    model="llama3.1:70b",
    graph_query=graph_query
)

behavioral_agent = BehavioralAgent(
    model="llama3.1:70b",
    graph_query=graph_query
)

mental_health_agent = MentalHealthAgent(
    model="meditron:70b",
    graph_query=graph_query
)

print("✓ Specialized agents initialized!")

## 5. Create Orchestrator

In [ ]:
orchestrator = OrchestratorAgent(
    model="llama3.1:70b",
    agents={
        "spatial": spatial_agent,
        "behavioral": behavioral_agent,
        "mental_health": mental_health_agent
    }
)

print("✓ Orchestrator initialized!")

## 6. Query the System

In [ ]:
def query_system(question: str):
    """Helper function to query the system and display results."""
    print(f"\n{'='*60}")
    print(f"Query: {question}")
    print(f"{'='*60}\n")
    
    message = AgentMessage(
        from_agent="user",
        to_agent="orchestrator",
        query=question
    )
    
    response = orchestrator.process_query(message)
    
    print(f"Response:\n{response.response}\n")
    print(f"Confidence: {response.confidence:.2%}")
    print(f"Execution Time: {response.execution_time:.2f}s")
    
    if response.metadata:
        print(f"Agents Consulted: {', '.join(response.metadata.get('agents_consulted', []))}")
    
    return response

### Example Query 1: Location Analysis

In [ ]:
response1 = query_system("What locations are most frequently visited by students?")

### Example Query 2: Mental Health Patterns

In [ ]:
response2 = query_system("How do activity patterns relate to mental health scores?")

### Example Query 3: Behavioral Analysis

In [ ]:
response3 = query_system("What behavioral patterns predict better mental health?")

## 7. System Status

In [ ]:
status = orchestrator.get_system_status()

print("\nSystem Status:")
print(f"\nOrchestrator: {status['orchestrator']['name']}")
print(f"Performance: {status['orchestrator']['performance']}")

print("\nSpecialized Agents:")
for agent_type, agent_status in status['agents'].items():
    print(f"  {agent_type}: {agent_status['model']}")

## Next Steps

1. **Build Knowledge Graph**: Run `python main.py --mode build-graph` to create the full knowledge graph from data
2. **Explore More Queries**: Try different questions combining spatial, behavioral, and mental health aspects
3. **Advanced Analytics**: Check out the analytics module for causal inference and predictions
4. **Visualizations**: Use the Gradio interface for interactive exploration

See `README.md` for complete documentation!